Create a csv file with the county FIPS code and the following 2017 census data:
- Land area in square kilometers
- Water area in square kilometers
- Farm area in square kilometers
- Crop area in square kilometers

In [ ]:
# geopandas not included in erdos environment, need to download
# Download Census TIGER/Line shapefules here: https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html

import geopandas as gpd
import pandas as pd

counties = gpd.read_file('../../work/tl_2017_us_county/tl_2017_us_county.shp')


In [ ]:
counties = counties.rename(
    columns = {
        'GEOID': 'FIPS',
    }
)

counties.head()

In [ ]:
xls = pd.ExcelFile('data/NASSAgcensusDownload2017.xlsx')

crop_df = pd.read_excel(xls, 'Farms', dtype={'FIPSTEXT': str})


In [ ]:
desired_columns = ['FIPSTEXT','y17_M059_valueNumeric', 'y17_M063_valueNumeric']

crop_df_acres = crop_df[desired_columns].rename(
    columns = {
        'FIPSTEXT': 'FIPS',
        'y17_M059_valueNumeric': 'farm_acre_as_percent',
        'y17_M063_valueNumeric': 'crop_acre_as_percent'
    }
)

In [ ]:
all_data = pd.merge(counties, crop_df_acres, on='FIPS')

all_data.head()

In [ ]:
county_area = all_data[["FIPS", "ALAND", 'AWATER', 'farm_acre_as_percent', 'crop_acre_as_percent']].copy()

county_area["area_land_km2"] = (
    county_area["ALAND"] / 1_000_000
)

county_area["area_water_km2"] = (
    county_area["AWATER"] / 1_000_000
)

county_area['farm_area_km2'] = (
    county_area['area_land_km2'] * county_area['farm_acre_as_percent'] / 100
)

county_area['crop_area_km2'] = (
    county_area['area_land_km2'] * county_area['crop_acre_as_percent'] / 100
)




In [ ]:
county_area.shape

In [ ]:
county_area.to_csv("../data/county_data_2017.csv", index=False)